# 1. Import Libraries

In [ ]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime

# 2. File Paths

In [ ]:
scripts_path = os.path.abspath(os.path.join('..', 'Scripts'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

data_dir = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
save_dir = os.path.abspath(os.path.join('..', 'Data', 'SavedModels'))
hyperparameters_dir = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))
visualization_dir = os.path.abspath(os.path.join('..', 'Data', 'Visualization'))
results_dir = os.path.abspath(os.path.join('..', 'Data', 'EvaluationResults'))

# 3. Load Models & Data

## 1. Data

### 1. Train Data

In [ ]:
# Load train data
train_data = np.load(os.path.join(data_dir, 'train_data.npy')).astype(np.float32)

num_users, num_items = train_data.shape
print(f"Dimensi Data: {num_users} Users, {num_items} Items")

### 2. Test Data

In [ ]:
# Load test data
test_data = np.load(os.path.join(data_dir, 'test_data.npy')).astype(np.float32)

# Mengambil nilai yang bukan 0, berarti yang sudah diberi rating oleh pengguna
test_indices = np.where(test_data > 0)

#### 1. Denormalize Test Data

In [ ]:
MAX_RATING = 5.0
# Denormalisasi nilai asli (karena data di-scaling dengan dibagi 5.0 di awal)
actual_values = test_data[test_indices] * MAX_RATING

### 3. z_mean Train
    Load fitur laten VAE yang disimpan saat Training — dibutuhkan oleh RSVDWithFeatures

In [ ]:
# Load z_mean hasil ekstraksi encoder VAE dari Training.ipynb
# Disimpan di data_dir (Data/TrainTest) bukan save_dir
z_mean_train = np.load(os.path.join(data_dir, 'z_mean_train.npy'))

print(f"Shape z_mean_train : {z_mean_train.shape}")
print(f"Shape train_data   : {train_data.shape}")

## 2. Models

In [ ]:
from model import Encoder, Decoder, VAE, RSVDWithFeatures

# 4. Construct Model

## 1. VAE

In [ ]:
# Load hyperparameter terbaik
vae_progress_file = os.path.join(hyperparameters_dir, 'tuning_progress_vae.json')
with open(vae_progress_file, 'r') as f:
    best_vae_params = json.load(f)['best_params']

# Membangun Arsitektur VAE
encoder = Encoder(hidden_dims=best_vae_params['hidden_dims'], latent_dim=best_vae_params['latent_dim'], dropout_rate=best_vae_params['dropout_rate'])
decoder = Decoder(hidden_dims=best_vae_params['hidden_dims'][::-1], output_dim=num_items)
eval_vae = VAE(encoder, decoder, beta=best_vae_params['beta'])

# Menyuntikkan Bobot ke VAE
_ = eval_vae(train_data[:1])
eval_vae.load_weights(os.path.join(save_dir, 'trained_best_vae_weights.weights.h5'))

## 2. RSVDWithFeatures

In [ ]:
# Load hyperparameter terbaik RSVD
rsvd_progress_file = os.path.join(hyperparameters_dir, 'tuning_progress_rsvd.json')
with open(rsvd_progress_file, 'r') as f:
    best_rsvd_params = json.load(f)['best_params']

# Inisialisasi RSVDWithFeatures dengan hyperparameter terbaik
eval_rsvd_fs = RSVDWithFeatures(
    n_factors=best_rsvd_params['n_factors'],
    learning_rate=best_rsvd_params['learning_rate'],
    lambda_reg=best_rsvd_params['lambda_reg']
)

# Muat bobot yang disimpan dari Training_FeatureStacking
# n_items disimpan terpisah agar predict() tahu batas kolom rating asli
eval_rsvd_fs.n_items = num_items
eval_rsvd_fs.mu     = np.load(os.path.join(save_dir, 'final_mu.npy'))
eval_rsvd_fs.b_u    = np.load(os.path.join(save_dir, 'final_b_u.npy'))
eval_rsvd_fs.b_i    = np.load(os.path.join(save_dir, 'final_b_i.npy'))
eval_rsvd_fs.U      = np.load(os.path.join(save_dir, 'best_U.npy'))
eval_rsvd_fs.Sigma  = np.load(os.path.join(save_dir, 'best_Sigma.npy'))
eval_rsvd_fs.V      = np.load(os.path.join(save_dir, 'best_V.npy'))

print(f"[INFO] Bobot RSVDWithFeatures berhasil dimuat")
print(f"[INFO] Shape U     : {eval_rsvd_fs.U.shape}")
print(f"[INFO] Shape V     : {eval_rsvd_fs.V.shape}")
print(f"[INFO] Shape Sigma : {eval_rsvd_fs.Sigma.shape}")

# 5. Prediction

## 1. VAE

### 1. Extract Latent Space

In [ ]:
# Ubah train data menjadi tensor
train_data_tf = tf.constant(train_data, dtype=tf.float32)

# Ekstrak matriks laten
Z_mean, Z_log_var = eval_vae.encoder.predict(train_data_tf, verbose=0)

### 2. Get Prediction

In [ ]:
# Prediksi dari Decoder (skala 0 - 1)
pred_vae_norm = eval_vae.decoder.predict(Z_mean, verbose=0)

### 3. Filter Prediction
    Filter hanya test data saja

In [ ]:
pred_vae_test = pred_vae_norm[test_indices]

### 4. Denormalize
    Kembalikan ke skala asli

In [ ]:
pred_vae_test_denorm = pred_vae_test * MAX_RATING

### 5. Clip Prediction
    Batasi prediksi agar tidak keluar dari batas rating

In [ ]:
pred_vae_clipped = np.clip(pred_vae_test_denorm, 1.0, 5.0)

## 2. RSVDWithFeatures

### 1. Get Prediction

In [ ]:
# Prediksi menggunakan z_mean_train sebagai fitur laten
# predict() secara internal menyusun ulang Z_aug dan memotong kolom laten dari output
full_rsvd_fs_pred = eval_rsvd_fs.predict(z_mean_train)

### 2. Filter Prediction

In [ ]:
# Filter untuk indeks test data saja
pred_rsvd_fs_test = full_rsvd_fs_pred[test_indices]

### 3. Denormalize

In [ ]:
# Kembalikan prediksi ke skala asli
pred_rsvd_fs_test_denorm = pred_rsvd_fs_test * MAX_RATING

### 4. Clip Prediction

In [ ]:
# Batasi prediksi
pred_rsvd_fs_clipped = np.clip(pred_rsvd_fs_test_denorm, 1.0, 5.0)

# 6. Evaluation

## 1. Helper Function

In [ ]:
# Fungsi helper untuk menghitung MSE, RMSE, MAE sekaligus
def calculate_metrics(y_true, y_pred):
    mse = np.mean(np.square(y_true - y_pred))
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    return mse, rmse, mae

## 2. Calculate Metrics

In [ ]:
mse_v,  rmse_v,  mae_v  = calculate_metrics(actual_values, pred_vae_clipped)       # VAE standalone
mse_fs, rmse_fs, mae_fs = calculate_metrics(actual_values, pred_rsvd_fs_clipped)   # RSVDWithFeatures

## 3. Comparison
    Bandingkan Feature Stacking vs hasil Weighted Average dari Testing.ipynb

In [ ]:
# Hasil weighted average dari Testing.ipynb (diisi manual untuk perbandingan)
# Ambil dari file final_testing_results_{timestamp}.json di EvaluationResults
WEIGHTED_AVG_MSE  = 0.8683
WEIGHTED_AVG_RMSE = 0.9318
WEIGHTED_AVG_MAE  = 0.7352

print(f"Metrik  | VAE Murni   | Feature Stacking | Weighted Average (ref)")
print("--------|-------------|------------------|-----------------------")
print(f"MSE     | {mse_v:.4f}      | {mse_fs:.4f}           | {WEIGHTED_AVG_MSE:.4f}")
print(f"RMSE    | {rmse_v:.4f}      | {rmse_fs:.4f}           | {WEIGHTED_AVG_RMSE:.4f}")
print(f"MAE     | {mae_v:.4f}      | {mae_fs:.4f}           | {WEIGHTED_AVG_MAE:.4f}")

## 4. Visualization
    Visualisasi perbandingan metrik Feature Stacking vs Weighted Average

In [ ]:
# Data perbandingan
model_labels  = ['VAE', 'Weighted Average', 'Feature Stacking']
rmse_values   = [rmse_v, WEIGHTED_AVG_RMSE, rmse_fs]
mae_values    = [mae_v,  WEIGHTED_AVG_MAE,  mae_fs]
colors        = ['#ff7f0e', '#1f77b4', '#2ca02c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

# ==========================================
# PLOT 1: RMSE
# ==========================================
ax1 = axes[0]
bars1 = ax1.bar(model_labels, rmse_values, color=colors, width=0.5)

# Tambahkan label nilai di atas setiap bar
for bar, val in zip(bars1, rmse_values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.set_title('Perbandingan RMSE', fontsize=14, fontweight='bold', pad=15)
ax1.set_ylabel('RMSE (lebih rendah lebih baik)', fontsize=12)
ax1.set_ylim(0, max(rmse_values) * 1.15)
ax1.grid(True, linestyle='--', alpha=0.5, axis='y')

# ==========================================
# PLOT 2: MAE
# ==========================================
ax2 = axes[1]
bars2 = ax2.bar(model_labels, mae_values, color=colors, width=0.5)

for bar, val in zip(bars2, mae_values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.set_title('Perbandingan MAE', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel('MAE (lebih rendah lebih baik)', fontsize=12)
ax2.set_ylim(0, max(mae_values) * 1.15)
ax2.grid(True, linestyle='--', alpha=0.5, axis='y')

plt.tight_layout()

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
plot_path = os.path.join(visualization_dir, f'feature_stacking_comparison_{timestamp}.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"\n[SUCCESS] Grafik perbandingan disimpan di: {plot_path}")
plt.show()

## 5. Save Results

In [ ]:
# Menyimpan seluruh ringkasan metrik agar bisa dipanggil kembali tanpa perlu me-run dari awal
evaluation_results = {
    "timestamp": timestamp,
    "architecture": "feature_stacking",
    "metrics": {
        "vae_standalone":     {"MSE": float(mse_v),  "RMSE": float(rmse_v),  "MAE": float(mae_v)},
        "rsvd_with_features": {"MSE": float(mse_fs), "RMSE": float(rmse_fs), "MAE": float(mae_fs)}
    },
    "reference_weighted_average": {
        "MSE":  WEIGHTED_AVG_MSE,
        "RMSE": WEIGHTED_AVG_RMSE,
        "MAE":  WEIGHTED_AVG_MAE
    }
}

evaluation_file_path = os.path.join(results_dir, f'feature_stacking_results_{timestamp}.json')

with open(evaluation_file_path, 'w') as f:
    json.dump(evaluation_results, f, indent=4)

print(f"\n[SUCCESS] Seluruh hasil evaluasi (Feature Stacking) telah diarsipkan ke:")
print(f"-> {evaluation_file_path}")